# Day 58：毕业项目开题选题与技术方案评审实战

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第六阶段（**Stage 6：企业级毕业项目实战与就业指导**）——**Day 58**！今天我们将：
> 1. 浏览全部 **8 大标准行业毕业赛道**（`CAPSTONE_TOPICS`）及其推荐架构与安全红线；
> 2. 使用 `TechnicalProposal` 编写结构化开题技术提案；
> 3. 运行 `review_proposal()` 进行 **RAG / Agent / Guardrails / Deployment** 四维自动化预审，验证完整提案通过与残缺提案拦截。

## 1. 环境初始化与导入 `project_iterations/day_58` 模块

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day58_dir = repo_root / "project_iterations" / "day_58"
if str(day58_dir) not in sys.path:
    sys.path.insert(0, str(day58_dir))

from zhishu_agent import (
    CAPSTONE_TOPICS,
    TechnicalProposal,
    create_sample_proposal,
    get_capstone_topic,
    list_capstone_topics,
    review_proposal,
)

print("已加载模块路径:", day58_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_58


## 2. 浏览 8 大行业毕业赛道（`CAPSTONE_TOPICS`）

In [2]:
topics = list_capstone_topics()
print(f"共收录 {len(topics)} 个毕业项目标准赛道:")
for idx, t in enumerate(topics, start=1):
    print(f"{idx}. {t.name} ({t.topic_id}) | 领域: {t.domain} | 推荐工具: {', '.join(t.recommended_tools)}")

assert len(CAPSTONE_TOPICS) == 8
assert get_capstone_topic("legal_assistant").name == "法律咨询助手"

共收录 8 个毕业项目标准赛道:
1. 法律咨询助手 (legal_assistant) | 领域: 法律合规 | 推荐工具: statute_search, contract_clause_diff, case_law_lookup
2. 简历优化Agent (resume_agent) | 领域: 人力资源与求职 | 推荐工具: pdf_resume_parser, jd_keyword_extractor, star_rewriter
3. 电商智能客服 (ecommerce_cs) | 领域: 新零售与电商 | 推荐工具: order_status_query, refund_eligibility_check, ticket_escalate
4. 论文阅读助手 (paper_reader) | 领域: 科研与学术 | 推荐工具: arxiv_fetcher, pdf_layout_parser, bibtex_formatter
5. 旅行规划Agent (travel_planner) | 领域: 文旅与出行 | 推荐工具: weather_forecast, poi_route_planner, budget_calculator
6. 企业财报分析系统 (financial_report) | 领域: 金融与投研 | 推荐工具: financial_ratio_calc, sec_filing_parser, chart_data_exporter
7. 教育答疑系统 (edu_tutor) | 领域: 智慧教育 | 推荐工具: knowledge_graph_lookup, step_verifier, quiz_generator
8. 医疗健康科普助手 (medical_science) | 领域: 大健康科普 | 推荐工具: lab_indicator_lookup, department_triage, emergency_red_flag_check


## 3. 完整开题提案四维评审（`APPROVED` 示例）

In [3]:
proposal_ok = create_sample_proposal("企业财报分析系统", author="陈默")
res_ok = review_proposal(proposal_ok)

print("项目名称:", res_ok.project_title)
print("评审状态:", res_ok.status, "| 总分:", res_ok.overall_score)
for dim, rev in res_ok.dimension_reviews.items():
    print(f"  - {dim}: {rev.score}/{rev.max_score} (passed={rev.passed})")
print("评审总结:", res_ok.summary)

assert res_ok.status == "APPROVED"
assert res_ok.overall_score == 100

项目名称: 智枢·企业财报分析系统系统
评审状态: APPROVED | 总分: 100
  - RAG: 25/25 (passed=True)
  - Agent: 25/25 (passed=True)
  - Guardrails: 25/25 (passed=True)
  - Deployment: 25/25 (passed=True)
评审总结: 提案《智枢·企业财报分析系统系统》四维架构完整（总分 100/100），准予开题进入 Day 59 开发阶段。


## 4. 残缺提案拦截与阻塞项诊断（`NEEDS_REVISION` 示例）

In [4]:
proposal_draft = TechnicalProposal(
    project_title="极速医疗问答",
    author="学员小李",
    topic_name="医疗健康科普助手",
    problem_statement="解答常见症状与体检报告指标疑问",
    target_users=["普通患者"],
    knowledge_sources=["卫健委临床指南"],
    chunking_strategy="按疾病章节语义切分",
    embedding_model="BAAI/bge-m3",
    agent_framework="LangGraph",
    tools=["lab_indicator_lookup", "emergency_red_flag_check"],
    memory_strategy="Redis 会话记忆",
    # 故意留空 Guardrails 与 Deployment 维度
)
res_draft = review_proposal(proposal_draft)

print("评审状态:", res_draft.status, "| 总分:", res_draft.overall_score)
print("阻塞项:")
for issue in res_draft.blocking_issues:
    print("  *", issue)
print("Guardrails 建议:", res_draft.dimension_reviews["Guardrails"].suggestions)

assert res_draft.status == "NEEDS_REVISION"
assert res_draft.overall_score == 50
assert res_draft.dimension_reviews["Guardrails"].passed is False
assert res_draft.dimension_reviews["Deployment"].passed is False

评审状态: NEEDS_REVISION | 总分: 50
阻塞项:
  * [Guardrails] 缺少: guardrails（安全护栏与合规策略）, eval_metrics（RAGAS / 业务准确率 / 延迟评测指标）
  * [Deployment] 缺少: model_serving（模型推理方案，如 vLLM / 量化模型 / API 网关）, deployment_stack（容器化与可观测性组件，如 Docker Compose / FastAPI）, budget_limit_cny（租户日成本预算与熔断阈值需 > 0）
Guardrails 建议: ['行业必备护栏：严禁下达临床诊断或开具处方; 急重症红旗症状立即提示就医; 医疗免责声明']
